# Level 5 — Data Mining Challenge: *The 1,000-Pick*

**규칙**: Set B (이미지 + 라벨 공개) 에서 **최대 1,000장** 을 선택하여 학습 셋에 추가하고, best 모델을 다시 학습하세요.

> **Set B 의 라벨이 공개되어 있다는 점에 주의**하세요. 본 Level 의 평가 본질은 "*주어진 풀에서 어떤 1,000장이 가장 가치 있는가*" — 즉, 라벨을 알고 있다고 가정한 상태에서의 효율적인 부분집합 선택입니다.

**본 PA에서 가장 큰 비중 (25%)** 을 차지하는 Level 입니다. 어떤 *알고리즘* 으로 1,000장을 골랐는지 — 그 *근거* — 가 변별력의 본진입니다. Curation Report 로 정리합니다.

채점 메트릭:
$$\text{DI} = \frac{\text{Avg-MF1}(\text{본인 picks}) - \text{Avg-MF1}(\text{random picks})}{\text{Avg-MF1}(\text{random picks})}$$

## 검토해 볼 만한 전략

| 전략 | 핵심 아이디어 | Set B 라벨 활용 |
|---|---|---|
| 클래스 균형 (Class Balancing) | Set A 에서 부족한 속성 클래스 (foggy / dawn-dusk 등) 를 채워 넣음 | ✅ 라벨로 직접 필터링 |
| Hard Example Mining | base 모델의 confidence 가 낮은 / 예측이 라벨과 다른 이미지를 우선 선택 | ✅ 모델 예측 vs 정답 비교 |
| 다양성 (Core-Set) | Set B 의 feature space 를 가장 잘 커버하는 부분집합 선택 (k-center / clustering) | 라벨 무관 |
| 결합 커버리지 | 속성 *조합* 의 균형을 맞춤 — 예: (snowy & night), (rainy & residential) | ✅ 라벨로 조합 카운트 |
| Loss 기반 | Set B 이미지에 대한 학습 직전 loss 가 큰 샘플 우선 | ✅ 라벨 필요 |

위 전략들을 결합/응용/대체할 수 있습니다. **Curation Report 에 본인의 의사결정 근거를 명확히 기술** 하세요.

**산출물**: `level5_picks.json` — 선택한 image_id 리스트 (이미지별 메타데이터 포함 가능).

In [ ]:
import os
import sys

repo_name = "2026-HYU-AUE8088-PA2"
if not os.path.exists(f"/content/{repo_name}"):
    !git clone https://github.com/jmk05764/2026-HYU-AUE8088-PA2.git
else:
    !git -C /content/{repo_name} fetch origin
    !git -C /content/{repo_name} reset --hard origin/main
    !git -C /content/{repo_name} log --oneline -1

%cd /content/{repo_name}

%load_ext autoreload
%autoreload 2

!pip install -q -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_CKPT_DIR = "/content/drive/MyDrive/aue8088-pa2/checkpoints"
import os; os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, CLASS_NAMES
from src.utils.submission import write_submission
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES, NUM_CLASSES
from src.models.resnet import resnet18
from src.models.vit import vit_small_patch16_224

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import wandb; wandb.login()   # API key 입력

WANDB_PROJECT = "aue8088-pa2"   # 비활성화하려면 None
STRATEGY_NAME = "top1k-uncertainty"   # 본인 전략명 (Run 이름에 들어감)
WANDB_TAGS    = ["level5", STRATEGY_NAME]

In [ ]:
# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨
DATA_ROOT = "../data/set_a"

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------


In [ ]:
# 1단계 — Level 3 best 모델(ViT-S pretrained+Focal+Sampler+Mix)로 Set B 전체 scoring.
import urllib.request

DEIT_PATH = "../checkpoints/deit_small_patch16_224.pth"
os.makedirs("../checkpoints", exist_ok=True)
if not os.path.exists(DEIT_PATH):
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/deit/deit_small_patch16_224-cd65a155.pth",
        DEIT_PATH)

def load_vit_l3(ckpt_path):
    m = vit_small_patch16_224().to(device)
    pre = torch.load(DEIT_PATH, map_location=device)["model"]
    def remap(sd):
        new = {}
        for k, v in sd.items():
            if k.startswith("head."): continue
            k = k.replace(".mlp.fc1.", ".mlp.0.").replace(".mlp.fc2.", ".mlp.3.")
            new[k] = v
        return new
    m.load_state_dict(remap(pre), strict=False)
    ckpt = torch.load(ckpt_path, map_location=device)
    m.load_state_dict(ckpt["state_dict"])
    return m

model = load_vit_l3(f"{DRIVE_CKPT_DIR}/level3_focal+sampler+Mix.pth")
model.eval()

set_b = BDDAttrDataset("../data/set_b", split="mining", transform=eval_transform())
loader_b = DataLoader(set_b, batch_size=64, shuffle=False, num_workers=2)

preds_b, probs_b, _, ids_b = collect_predictions(model, loader_b, device)

# 이미지별 불확실성: 1 - max-softmax 를 3개 head 평균
max_probs = np.stack([probs_b[a].max(axis=-1) for a in ATTRIBUTES], axis=1)
uncertainty = 1.0 - max_probs.mean(axis=1)
print(f"unc shape: {uncertainty.shape}, mean={uncertainty.mean():.3f}")

In [ ]:
import json
from collections import Counter

K = 1000

# ── Set A 클래스별 빈도 → 희소성 가중치 ──────────────────────────────────────
train_ds = BDDAttrDataset("../data/set_a", "train", transform=eval_transform())

inv_freq = {}
for attr in ATTRIBUTES:
    counts = train_ds.class_counts(attr).float()
    w = 1.0 / counts.clamp(min=1)
    inv_freq[attr] = (w / w.sum()).numpy()

print("Set A 클래스 분포:")
for attr in ATTRIBUTES:
    counts = train_ds.class_counts(attr).tolist()
    print(f"  {attr}: { {CLASS_NAMES[attr][i]: int(c) for i, c in enumerate(counts)} }")

# ── 3가지 신호 계산 ────────────────────────────────────────────────────────────
N = len(set_b)

# 1) rarity: Set A에서 해당 클래스가 희소할수록 높음
rarity = np.array([
    np.mean([inv_freq[a][getattr(set_b.samples[i], a)]
             for a in ATTRIBUTES if getattr(set_b.samples[i], a) >= 0])
    for i in range(N)
])

# 2) error: 현재 모델이 틀린 속성 비율 (0.0~1.0)
error = np.array([
    np.mean([int(preds_b[a][i] != getattr(set_b.samples[i], a))
             for a in ATTRIBUTES if getattr(set_b.samples[i], a) >= 0])
    for i in range(N)
])

# 3) uncertainty: 1단계에서 이미 계산 완료

# 결합 점수 (가중합)
score = 0.4 * rarity + 0.3 * uncertainty + 0.3 * error
print(f"\n신호 평균: rarity={rarity.mean():.4f}  uncertainty={uncertainty.mean():.4f}  error={error.mean():.4f}")

# ── Phase 1: 희소 클래스 강제 할당 (Critical Quota) ──────────────────────────
# Level 4 Confusion Matrix 기준 recall 순으로 우선순위 결정
# foggy=0.00, snowy=0.34, residential=0.49, dawn/dusk=0.54
QUOTAS = [
    ("weather",   4, 200),   # foggy       — recall 0.00 (전량 오분류)
    ("weather",   3, 150),   # snowy       — recall 0.34
    ("timeofday", 2, 150),   # dawn/dusk   — recall 0.54
    ("scene",     2, 100),   # residential — recall 0.49
]

selected = set()
for attr, cls_idx, quota in QUOTAS:
    cands = [i for i in range(N)
             if getattr(set_b.samples[i], attr) == cls_idx and i not in selected]
    # 같은 클래스 내에서도 score 높은 순으로 선택
    cands_sorted = sorted(cands, key=lambda i: -score[i])
    added = cands_sorted[:quota]
    selected.update(added)
    label_name = CLASS_NAMES[attr][cls_idx]
    print(f"  [{attr}={label_name}] 가용 {len(cands)}장 중 {len(added)}장 할당")

print(f"\nPhase 1 완료: {len(selected)}장 확보")

# ── Phase 2: 나머지 슬롯을 score 순으로 채우기 ────────────────────────────────
remaining = [i for i in np.argsort(-score) if i not in selected]
fill = remaining[: K - len(selected)]
selected.update(fill)
print(f"Phase 2 완료: {len(fill)}장 추가 → 총 {len(selected)}장")

# ── score 기준 내림차순 정렬 ───────────────────────────────────────────────────
all_picks_sorted = sorted(list(selected), key=lambda i: -score[i])

STRATEGY_DESC = (
    "Class-Aware Multi-Signal Curation: "
    "Level 4 Confusion Matrix 기반 희소 클래스(foggy/snowy/dawn-dusk/residential)에 "
    "우선 할당 후, rarity(0.4)+uncertainty(0.3)+error(0.3) 결합 점수로 나머지 슬롯을 채움."
)

def make_picks_list(indices):
    return [
        {
            "image_id":  set_b.samples[i].image_id,
            "weather":   int(set_b.samples[i].weather),
            "scene":     int(set_b.samples[i].scene),
            "timeofday": int(set_b.samples[i].timeofday),
            "score":     round(float(score[i]), 6),
        }
        for i in indices
    ]

# Ablation용 서브셋 저장 (250 ⊆ 500 ⊆ 1000)
for k in [250, 500, 1000]:
    subset = make_picks_list(all_picks_sorted[:k])
    with open(f"../level5_picks_{k}.json", "w") as f:
        json.dump({"strategy": STRATEGY_DESC, "num_picks": len(subset), "picks": subset}, f, indent=2)
    print(f"저장: level5_picks_{k}.json")

# 기본 파일
picks = make_picks_list(all_picks_sorted[:K])
with open("../level5_picks.json", "w") as f:
    json.dump({"strategy": STRATEGY_DESC, "num_picks": len(picks), "picks": picks}, f, indent=2)

# ── picks 분포 출력 ───────────────────────────────────────────────────────────
print("\n선택된 1000장 분포:")
for attr in ATTRIBUTES:
    cnt = Counter(p[attr] for p in picks)
    print(f"  {attr}: { {CLASS_NAMES[attr][k]: cnt.get(k,0) for k in range(NUM_CLASSES[attr])} }")


# Google Drive 백업 (세션 끊겨도 유지)
import shutil
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
for k in [250, 500, 1000]:
    shutil.copy(f"../level5_picks_{k}.json",
                f"{DRIVE_CKPT_DIR}/level5_picks_{k}.json")
shutil.copy("../level5_picks.json", f"{DRIVE_CKPT_DIR}/level5_picks.json")
print("Drive 저장 완료:", DRIVE_CKPT_DIR)

In [ ]:
# 3단계 — Set A + 본인이 고른 picks 로 재학습. 학습 메트릭은 wandb 로 자동 로깅.
extra = [(p["image_id"], p["weather"], p["scene"], p["timeofday"]) for p in picks]
train_aug = BDDAttrDataset("../data/set_a", "train", transform=train_transform(), extra_picks=extra)
val_ds    = BDDAttrDataset("../data/set_a", "val",   transform=eval_transform())

g = torch.Generator(); g.manual_seed(SEED)
loader_tr  = DataLoader(train_aug, batch_size=64, shuffle=True,  num_workers=2, worker_init_fn=seed_worker, generator=g, pin_memory=True)
loader_val = DataLoader(val_ds,    batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

set_seed(SEED, deterministic=True)
model2 = load_vit_l3(f"{DRIVE_CKPT_DIR}/level3_focal+sampler+Mix.pth")
optim  = torch.optim.AdamW(model2.parameters(), lr=5e-4, weight_decay=5e-2)
sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=25, eta_min=1e-6)
losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}

logger = WandbLogger(
    project=WANDB_PROJECT,
    run_name=f"level5-{STRATEGY_NAME}",
    config={
        "backbone": "vit_small_patch16_224", "strategy": STRATEGY_NAME,
        "num_picks": len(picks), "epochs": 25, "batch": 64, "lr": 3e-4, "seed": SEED,
    },
    tags=WANDB_TAGS,
)
trainer = MultiTaskTrainer(model2, optim, sched, losses, device, TrainConfig(epochs=25), logger=logger)
history = trainer.fit(loader_tr, loader_val)

# 학습 종료 후 — 속성별 confusion matrix 와 picks 분포를 wandb 에 업로드
val_pred, _, val_tgt, _ = collect_predictions(model2, loader_val, device)
for a in ATTRIBUTES:
    logger.log_confusion_matrix(f"final/cm_{a}", confusion_matrices(val_pred, val_tgt)[a], CLASS_NAMES[a])

# 본인이 고른 1,000장의 (예측) 분포 — Curation 의도가 의도대로 반영됐는지 검증
from collections import Counter
for a in ATTRIBUTES:
    cnt = Counter(p[a] for p in picks)
    rows = [[CLASS_NAMES[a][k], cnt.get(k, 0)] for k in range(NUM_CLASSES[a])]
    logger.log_table(f"picks/distribution_{a}", ["class", "count"], rows)

torch.save({"state_dict": model2.state_dict(), "history": history},
           "../checkpoints/level5_final.pth")

# Google Drive 백업
import shutil
shutil.copy("../checkpoints/level5_final.pth", f"{DRIVE_CKPT_DIR}/level5_final.pth")
print(f"Drive 저장 완료: {DRIVE_CKPT_DIR}/level5_final.pth")
logger.finish()

In [ ]:
# 4단계 — Kaggle 제출용 CSV 생성.
test_ds = BDDAttrDataset("../data/set_a", "test", transform=eval_transform())
loader_te = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=2)

preds_te, _, _, ids_te = collect_predictions(model2, loader_te, device)
write_submission("../submission/level5_submission.csv", ids_te, preds_te)
print("submission/level5_submission.csv 생성 완료 — Kaggle 페이지에 직접 업로드 하세요.")

## Curation Report — 필수

Final PPT 에 다음을 포함하세요.
- **선별 알고리즘** (의사코드 또는 1페이지 다이어그램).
- 본인 picks 1,000장의 **분포** — (예측된) weather × scene × timeofday — 를 heatmap 또는 stacked bar 로 시각화.
- **Random-1000 baseline** 결과와 본인의 **DI score** 비교.
- **Ablation**: 250 / 500 / 1000 장을 골랐을 때의 변화 — 추가 데이터의 한계 효용이 보이는지 확인.

여러 전략을 시험했다면 wandb 의 같은 프로젝트에 `STRATEGY_NAME` 만 바꿔서 별도 Run 으로 누적하세요. 학습 곡선·분포·DI score 비교가 한 페이지에 모입니다.